
# Comparação de Modelos TPP: RoTHP vs HoTHP vs THP

Este notebook compara o desempenho de três modelos de Processos de Ponto Temporal (TPP) baseados em Transformer:
1. **THP (Transformer Hawkes Process):** Usa Positional Encoding temporal (sinusoidal) padrão.
2. **RoTHP (Rotary THP):** Usa Rotary Positional Encoding (RoPE) adaptado para tempo contínuo.
3. **HoTHP (Hyperbolic RoTHP):** Nossa proposta, usa embeddings hiperbólicos para garantir decaimento monotônico da atenção.

O objetivo é demonstrar que o HoTHP é superior em cenários de **longa dependência** e **extrapolação**.


In [ ]:

import sys
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

try:
    import google.colab
    if not os.path.exists('/content/ufc-easytpp'):
        !git clone https://github.com/hugoramos/ufc-easytpp.git
    project_root = '/content/ufc-easytpp'
except:
    project_root = os.getcwd()
    if os.path.basename(project_root) == 'notebooks':
        project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Patch FP16 para estabilidade numérica
import easy_tpp.model.torch_model.torch_baselayer as baselayer
def attention_fixed(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        # Corrige para usar -1e4 em vez de -1e9 para evitar NaN em FP16
        scores = scores.masked_fill(mask > 0, -1e4)
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn
baselayer.attention = attention_fixed

# Import Models
from easy_tpp.model.torch_model.torch_thp import THP
from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Hardware: {device}")


In [ ]:

# === Geração de Dados Sintéticos (Hawkes Process) ===
import math

def generate_hawkes_data(num_seqs, max_len, mu=0.1, alpha=0.5, beta=1.0):
    print(f"Gerando {num_seqs} sequências Hawkes (len={max_len})...")
    data = []
    
    # Adicionando um token de "BOS" (Begin of Sequence) implícito no tempo 0
    # O primeiro evento real será t1 > 0
    
    for _ in tqdm(range(num_seqs), desc="Simulando"):
        t = 0
        hist = []
        timestamps = [0.0] # Começa em 0.0 para cálculo de delta
        types = [0] # Tipo 0 = Padding/Start
        
        while len(timestamps) < max_len:
            # Lambda(t) = mu + sum(alpha * exp(-beta * (t - ti)))
            # Upper bound lambda_bar (no tempo t atual)
            decay_term = sum([math.exp(-beta * (t - ti)) for ti in hist])
            lambda_bar = mu + alpha * decay_term
            
            # Amostra próximo tempo candidato (Poisson homogêneo com lambda_bar)
            w = -math.log(np.random.uniform()) / lambda_bar
            t += w
            
            # Acceptance step (Thinning Algorithm)
            decay_term_new = sum([math.exp(-beta * (t - ti)) for ti in hist])
            lambda_t = mu + alpha * decay_term_new
            
            if np.random.uniform() * lambda_bar <= lambda_t:
                timestamps.append(t)
                types.append(1) # Tipo 1 = Evento genérico
                hist.append(t)
                
        # Padding se necessário (aqui fixamos o tamanho exato no while, mas garantindo tensores)
        # O modelo espera:
        # time_seqs: [0.0, t1, t2, ..., tn]
        # delta_seqs: [0.0, t1-0, t2-t1, ...]
        
        time_tensor = torch.tensor(timestamps[:max_len], dtype=torch.float32)
        type_tensor = torch.tensor(types[:max_len], dtype=torch.long)
        
        # Calcular deltas
        deltas = torch.zeros_like(time_tensor)
        deltas[1:] = time_tensor[1:] - time_tensor[:-1]
        
        data.append({
            'time_seqs': time_tensor,
            'time_delta_seqs': deltas,
            'type_seqs': type_tensor
        })
    return data

# Batch Loader customizado para os dicionários gerados
def get_batch(data, batch_size=16, device=device):
    indices = np.random.choice(len(data), batch_size)
    batch = [data[i] for i in indices]
    
    # Stack tensors
    time_seqs = torch.stack([b['time_seqs'] for b in batch]).to(device)
    delta_seqs = torch.stack([b['time_delta_seqs'] for b in batch]).to(device)
    type_seqs = torch.stack([b['type_seqs'] for b in batch]).to(device)
    
    # Mask para THP/RoTHP (1 = evento válido, 0 = padding)
    # Como geramos tamanho fixo, tudo é 1 (exceto se implementássemos padding variável)
    # O attention mask é [batch, seq, seq] triangular ou [batch, seq] dependendo da implementação base.
    # No EasyTPP, 'attention_mask' é boolean [batch, seq, seq].
    
    seq_len = time_seqs.size(1)
    # Máscara triangular para causalidade (True = Masked/Blocked, False = Visible)
    # EasyTPP usa: True = mask (bloqueado), False = não mask (visível)?
    # Verificando baselayer: scores.masked_fill(mask > 0, -1e4). 
    # Então se mask > 0, ele bloqueia.
    
    # Vamos criar máscara causal padrão:
    # Triângulo superior (excluindo diagonal) deve ser bloqueado (futuro não vê passado)
    # triu(ones, k=1) -> 1 no futuro.
    attn_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).to(device).bool()
    attn_mask = attn_mask.unsqueeze(0).expand(batch_size, -1, -1)
    
    # Batch Non Pad Mask [batch, seq]
    batch_non_pad_mask = torch.ones_like(type_seqs).bool().to(device)
    
    return time_seqs, delta_seqs, type_seqs, batch_non_pad_mask, attn_mask


In [ ]:

from easy_tpp.config_factory.model_config import ModelConfig
from easydict import EasyDict

# === Parâmetros do Experimento ===
SEQ_LEN_TRAIN = 50
SEQ_LEN_TEST = 200 # Extrapolação!
NUM_TRAIN = 500
NUM_TEST = 100

print("1. Gerando Dados...")
train_data = generate_hawkes_data(NUM_TRAIN, SEQ_LEN_TRAIN)
test_data = generate_hawkes_data(NUM_TEST, SEQ_LEN_TEST)

# Configuração Comum
base_config = {
    'hidden_size': 64,
    'num_layers': 2,
    'num_heads': 4,
    'dropout_rate': 0.1,
    'num_event_types': 2,      # 0:start, 1:event
    'num_event_types_pad': 2,  # total vocab size
    'event_pad_index': 0,      # usando 0 como pad/start
    'time_emb_size': 64,       # usado pelo THP
    'use_ln': True,
    'gpu': 0 if torch.cuda.is_available() else -1,
    'model_id': 'Generic',
    'thinning': None,          # Sem amostragem complexa no treino por enquanto
    'loss_integral_num_sample_per_step': 20,
    'use_mc_samples': False    # Trapezoidal approximation
}

config = ModelConfig(**base_config)

# Instanciando Modelos
models = {
    'THP (Seno)': THP(config).to(device),
    'RoTHP (Trig)': RoTHP(config).to(device),
    'HoTHP (Hyper)': HoTHP(config).to(device)
}

# Histórico de Loss
history = {k: {'train': [], 'test': []} for k in models.keys()}

print("\n2. Iniciando Treinamento Comparativo...")

EPOCHS = 100
BATCH_SIZE = 32

for name, model in models.items():
    print(f"\n>>> Treinando {name}...")
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    for epoch in range(EPOCHS):
        model.train()
        
        # Batch aleatório
        ts, dts, types, np_mask, attn_mask = get_batch(train_data, BATCH_SIZE)
        batch = (ts, dts, types, np_mask, attn_mask)
        
        loss, num_events = model.loglike_loss(batch)
        nll = loss / num_events
        
        opt.zero_grad()
        nll.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        
        history[name]['train'].append(nll.item())
        
        # Avaliação Periódica (Extrapolação)
        if epoch % 10 == 0 or epoch == EPOCHS - 1:
            model.eval()
            with torch.no_grad():
                # Pega todo o teste (ou batch grande)
                ts_t, dts_t, types_t, np_mask_t, attn_mask_t = get_batch(test_data, batch_size=50)
                batch_test = (ts_t, dts_t, types_t, np_mask_t, attn_mask_t)
                
                loss_t, num_t = model.loglike_loss(batch_test)
                nll_t = loss_t / num_t
                history[name]['test'].append(nll_t.item())
            
            print(f"Ep {epoch}: Train NLL={nll.item():.4f} | Test NLL (Extrap)={nll_t.item():.4f}")



In [ ]:

# === Visualização ===

plt.figure(figsize=(12, 5))

# Plot Test NLL (Capacidade de Extrapolação)
final_results = {}

for name in models.keys():
    plt.plot(history[name]['test'], label=name, marker='o')
    final_results[name] = history[name]['test'][-1]

plt.title('Performance de Extrapolação (Test NLL em Sequências Longas)')
plt.ylabel('Negative Log-Likelihood (Menor é Melhor)')
plt.xlabel('Épocas (x10)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n=== Resultados Finais (NLL no Teste Longo) ===")
for k, v in final_results.items():
    print(f"{k}: {v:.4f}")

best_model = min(final_results, key=final_results.get)
print(f"\n🏆 Vencedor: {best_model}")
